# 04 — Budget Prediction (XGBoost Regressor)

**Goal:** Train an XGBoost Regressor to predict the `ContractCost` based on categorical features and historical data.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 1. Load Preprocessed Data

In [ ]:
df = pd.read_parquet('../model_artifacts/processed_data.parquet')
config = joblib.load('../model_artifacts/feature_config.pkl')

budget_features = config['budget_features']
budget_target = config['budget_target']

print(f'Features ({len(budget_features)}): {budget_features}')
print(f'Target: {budget_target}')

## 2. Train/Test Split

In [ ]:
X = df[budget_features]
y = df[budget_target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')

## 3. Train XGBoost Regressor

In [ ]:
reg = xgb.XGBRegressor(
    n_estimators=150,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mae'
)

reg.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=10
)

## 4. Evaluation

In [ ]:
y_pred = reg.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print('=== Regression Metrics ===')
print(f'MAE:  PHP {mae:,.2f}')
print(f'RMSE: PHP {rmse:,.2f}')
print(f'R2:   {r2:.4f}')

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5, color='coral')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Actual vs Predicted Contract Cost', fontweight='bold')
plt.xlabel('Actual Cost (PHP)')
plt.ylabel('Predicted Cost (PHP)')
plt.tight_layout()
plt.show()

## 5. Feature Importance

In [ ]:
importance = pd.DataFrame({
    'Feature': budget_features,
    'Importance': reg.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance, x='Importance', y='Feature', palette='magma')
plt.title('Feature Importance (Budget Prediction)', fontsize=14, fontweight='bold')
plt.show()

## 6. Export Model Artifact

In [ ]:
model_path = '../model_artifacts/xgb_budget_model.pkl'
joblib.dump(reg, model_path)
print(f'Saved budget model to {model_path}')